# Лабораторная работа №4
## Числовые характеристики выборки с помощью встроенных функций

**Вариант:** `16 % 10 = 6`

Цель работы: научиться вычислять числовые характеристики выборки по формулам, которые обычно реализуются встроенными функциями Excel.


## Краткая теория

Для выборки $x_1, \dots, x_n$ используются следующие характеристики:

- среднее $\bar x$;
- медиана;
- мода;
- стандартное отклонение $s$;
- дисперсия $s^2$;
- асимметрия;
- эксцесс;
- минимум и максимум;
- размах $R = x_{\max} - x_{\min}$;
- сумма и объём выборки;
- стандартная ошибка среднего;
- 95% доверительный интервал для среднего.

В Python те же величины удобно вычислять напрямую по формулам, а не через Excel-функции.


In [ ]:
import math
import statistics as st
from collections import Counter
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
})

def sample_stats(sample):
    arr = np.asarray(sample, dtype=float)
    n = arr.size
    mean = float(np.mean(arr))
    median = float(np.median(arr))
    modes = sorted(st.multimode(arr.tolist()))
    variance = float(np.var(arr, ddof=1))
    std_dev = float(np.sqrt(variance))
    stderr = std_dev / math.sqrt(n)
    if n > 2 and std_dev > 0:
        skew = float(n / ((n - 1) * (n - 2)) * np.sum(((arr - mean) / std_dev) ** 3))
    else:
        skew = float("nan")
    if n > 3 and std_dev > 0:
        excess = float(
            n * (n + 1) / ((n - 1) * (n - 2) * (n - 3)) * np.sum(((arr - mean) / std_dev) ** 4)
            - 3 * (n - 1) ** 2 / ((n - 2) * (n - 3))
        )
    else:
        excess = float("nan")
    ci_half = 1.96 * stderr
    return {
        "mean": mean,
        "median": median,
        "modes": modes,
        "std_dev": std_dev,
        "variance": variance,
        "skew": skew,
        "excess": excess,
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "range": float(np.max(arr) - np.min(arr)),
        "sum": float(np.sum(arr)),
        "stderr": stderr,
        "ci": (mean - ci_half, mean + ci_half),
    }

def draw_hist(sample, filename):
    arr = np.asarray(sample, dtype=float)
    bins = max(1, int(round(math.sqrt(arr.size))))
    edges = np.linspace(arr.min(), arr.max(), bins + 1)
    counts, _ = np.histogram(arr, bins=edges)
    mids = (edges[:-1] + edges[1:]) / 2
    densities = counts / (arr.size * np.diff(edges))
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(mids, densities, width=np.diff(edges), align="center", color="#5B8FF9", edgecolor="black")
    ax.axvline(arr.mean(), color="#D94F70", linewidth=2, label="Среднее")
    ax.axvline(np.median(arr), color="#3A9D5D", linewidth=2, linestyle="--", label="Медиана")
    ax.set_xlabel("x")
    ax.set_ylabel("Плотность относительной частоты")
    ax.set_title("Гистограмма выборки")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(filename, bbox_inches="tight")
    plt.close(fig)

sample = [16.3, 20.6, 19.4, 18.7, 16.3, 18.7, 19.3, 18.8, 21.8, 23.2, 22.7, 17.4, 21.8, 18.8, 20.2, 19.3, 19.4, 18.4, 19.3, 18.1, 19.4, 19.7, 21.8, 18.8]
stats = sample_stats(sample)
draw_hist(sample, "figures/lab4_hist.png")
print(stats)


## Результаты вычислений

| Показатель             | Значение                           |
| ---------------------- | ---------------------------------- |
| Объём выборки          | 24                                 |
| Среднее                | 19.5083                            |
| Медиана                | 19.3000                            |
| Моды                   | 18.8000, 19.3000, 19.4000, 21.8000 |
| Стандартное отклонение | 1.7754                             |
| Дисперсия              | 3.1521                             |
| Асимметрия             | 0.3292                             |
| Эксцесс                | 0.0465                             |
| Минимум                | 16.3000                            |
| Максимум               | 23.2000                            |
| Размах                 | 6.9000                             |
| Сумма                  | 468.2000                           |
| Стандартная ошибка     | 0.3624                             |
| 95% ДИ для среднего    | [18.7980; 20.2186]                 |

![Гистограмма выборки](figures/lab4_hist.png)


## Вывод

Для варианта 6 получен полный набор основных описательных характеристик выборки. При наличии нескольких мод удобнее явно перечислять все значения, которые встречаются с максимальной частотой.
